# 04. Retrieval 설계: 필요한 데이터를 잘 가져오는 법

**가져온 근거가 틀리면 답은 틀립니다.** 문장 다섯 개로 점수를 손으로 계산하고, 어려운 질문으로 방식을 **숫자로 비교**한 뒤, 재순위, Decision API 관문(분해, 출처 보강), PDF 쪽 판독까지 재 봅니다.

## 학습 목표

- 토큰, BM25, 코사인, RRF를 손으로 계산해 코드와 맞추고, 청킹과 범위 필터가 결과를 바꾸는 것을 보인다.
- 어려운 질문에서 방식별 recall을 재고 누락과 오인을 구분하며, 표본이 작으면 작다고 말한다.
- 후보 깊이가 recall의 천장인 이유와 벤치마크 승자를 그대로 믿으면 안 되는 이유를 수치로 말한다.
- 분해가 Decision API 관문 뒤에서 기본으로 켜진 이유와 분해, 출처 보강의 측정 효과를 구분해 말한다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 왜 여전히 RAG인가, 근거 단위 | 3 |
| 2 | 다섯 문장의 토큰 | 3 |
| 3 | BM25의 세 요소 | 4 |
| 4 | Dense와 코사인 유사도 | 3 |
| 5 | RRF: 점수가 아니라 순위 | 3 |
| 6 | 청킹 규칙, 메타데이터는 자격 | 3 |
| 7 | 범위 필터 | 3 |
| 8 | 측정: recall, 누락과 오인 | 2 |
| 9 | 어려운 질문으로 측정 | 5 |
| 10 | 넓은 후보와 원 질문 재순위 | 7 |
| 11 | Jev 관문: 분해와 출처 보강 | 6 |
| 12 | AutoRAG와 다른 점 | 3 |
| 13 | PDF 쪽 판독 | 5 |
| - | 직접 해보기 | 5 |
| - | 확인 질문 | 3 |
| - | 정리 | 2 |

## 호출 규모

Claude 호출은 세 곳(열린 관문 뒤 질문 분해, PDF 쪽 판독, 내 질문 분해)에 각 최대 1회입니다. Decision API(Jev) 호출은 관문 셀 3회와 1회, 직접 해보기 1회이고 질문 문장이 공급자로 전송됩니다. 호출 직전에 규모를 고지합니다. 임베딩과 재순위는 로컬 모델이라 API 호출이 아니지만 **처음 쓸 때 파일(약 470MB, 2.1GB)을 내려받습니다.**

표지: `[따라하기]` 그대로 실행, `[바꿔보기]` **결과를 먼저 예측**하고 값 변경, `[깨뜨리기]` 일부러 실패 읽기, `[내 것 만들기]` 내 것으로 바꾸기.

In [ ]:
from datetime import date
from math import log
from time import perf_counter

import numpy as np
import pymupdf
from kiwipiepy import Kiwi

from rag_lab.decisions import make_decider
from rag_lab.corpus import Chunk, Scope, count_cut_tables, fixed_chunks, load_cases, load_corpus, section_chunks
from rag_lab.evaluation import evaluate_retrieval, load_gold, recall_by_type
from rag_lab.evidence import RetrievalContext
from rag_lab.gates import needs_decomposition
from rag_lab.pages import needs_reading, read_page
from rag_lab.pipeline import decompose_question, retrieve
from rag_lab.retrieval import BM25Index, Retriever, rrf, tokenize
from rag_lab.settings import ROOT, announce, key_status, make_embeddings, make_model, make_reranker, require_key

require_key()  # 키가 없으면 여기서 멈추고 .env에 채우는 방법을 알려 줍니다.
print(key_status())

## 1. 왜 여전히 RAG인가, 근거 단위 (3분)

모델은 **우리 회사 문서를 모릅니다.** 문서를 전부 넣으면 비용이 들고 옛 버전과 초안이 섞입니다. **필요한 것만, 자격 있는 것만, 인용할 수 있는 모양으로** 가져오는 일은 모델 바깥의 설계입니다. 먼저 정할 것은 **근거 하나가 무엇이냐**입니다. 그 조각만 읽어도 뜻이 통해야 인용할 수 있습니다.

## 2. 다섯 문장의 토큰 (3분)

검색은 글자가 아니라 **토큰**을 맞춥니다. 조사가 붙은 `식당은`과 `식당을`은 공백 분리로는 다른 낱말이지만, 형태소 분석기 Kiwi는 조사를 떼어 줍니다. 아래 셀은 무료입니다.

In [ ]:
# [따라하기] 공백 분리와 Kiwi 토큰 비교 (무료, 모델 호출 0회)
SENTENCES = ["사내 식당은 오전 11시 30분에 문을 엽니다.", "도서관 책은 2주 동안 빌릴 수 있습니다.",
             "회의실은 하루 전까지 예약해야 합니다.", "식당 메뉴는 매주 월요일에 공지합니다.",
             "통근 버스는 오후 6시에 출발합니다."]

kiwi = Kiwi()
print("공백 분리:", SENTENCES[0].split())
print("Kiwi    :", [(t.form, t.tag) for t in kiwi.tokenize(SENTENCES[0])])

**관찰할 것.** Kiwi는 `식당은`을 `식당`과 `은`으로 나눕니다. `NNG`는 명사, `JX`는 조사입니다. 검색 토큰은 `tokenize`가 만들며 명사, 동사, 영문, 숫자만 남깁니다.

**[바꿔보기]** 문장을 `식당을`, `식당에서`로 바꿉니다. **예측 먼저:** 세 형태의 `tokenize` 결과는 같을까요, 다를까요? 적어 본 뒤 실행하세요.

In [ ]:
# [바꿔보기] 바꿀 값: VARIANTS의 문장. 예측을 먼저 적고 실행하세요. (무료)
VARIANTS = ["식당은 열려 있나요", "식당을 찾습니다", "식당에서 먹습니다"]

for text in VARIANTS:
    print(f"{text!r:18} 공백 분리={text.split()}  tokenize={tokenize(text)}")

## 3. BM25의 세 요소 (4분)

- **IDF**: 적은 문장에만 나오는 낱말일수록 큽니다. `log(1 + (N - df + 0.5) / (df + 0.5))`로 항상 0보다 큽니다(N 조각 수, df 그 낱말이 든 조각 수).
- **TF**: 문장 안 등장 횟수. 늘수록 점수가 오르지만 완만합니다.
- **길이 보정**: 평균보다 긴 문장은 점수를 깎습니다.

In [ ]:
# [따라하기] IDF를 손으로 계산해 BM25Index와 맞추고 검색한다 (무료)
chunks = [Chunk(f"s{i}", "notes", "", s, 0, len(s), 1, 1) for i, s in enumerate(SENTENCES)]
index = BM25Index(chunks)

N = len(chunks)
df = sum("식당" in tokenize(c.text) for c in chunks)
by_hand = log(1 + (N - df + 0.5) / (df + 0.5))
print(f"N={N}, df(식당)={df}, 손계산 idf={by_hand:.4f}, 색인 idf={index.idf['식당']:.4f}")

for chunk_id, score in index.search("식당 메뉴 언제 공지해?", limit=3):
    print(chunk_id, round(score, 3), SENTENCES[int(chunk_id[1:])])

**관찰할 것.** 손계산과 색인 값이 일치합니다. 질의 낱말이 많이 겹친 문장이 위에 있고, 하나만 겹친 문장은 아래입니다.

**[바꿔보기]** 질의를 `대출 기간`으로 바꿉니다. 문장에는 `대출`이라는 낱말이 없습니다. **예측 먼저:** 도서관 문장이 나올까요?

In [ ]:
# [바꿔보기] 바꿀 값: QUERY. 예측을 먼저 적고 실행하세요. (무료)
QUERY = "대출 기간"

print("질의 토큰:", tokenize(QUERY))
print("BM25 결과:", index.search(QUERY, limit=3))

**관찰할 것.** 결과가 비어 있다면 질의 토큰이 어느 문장에도 없기 때문입니다. **낱말이 다르면** BM25는 후보를 만들지 못하고, 그래서 dense가 필요합니다.

`BM25Okapi`는 낱말이 문서 절반 넘게 들어 있으면 IDF가 음수가 됩니다. 문장 한 개로 확인합니다.

In [ ]:
# [깨뜨리기] 문장 하나만 넣은 BM25Okapi와 BM25Index를 비교한다 (무료)
from rank_bm25 import BM25Okapi

one = SENTENCES[3]
okapi = BM25Okapi([tokenize(one)])
mine = BM25Index([Chunk("one", "notes", "", one, 0, len(one), 1, 1)])

query = "식당 메뉴"
okapi_scores = okapi.get_scores(tokenize(query))
print("Okapi idf :", {t: round(v, 3) for t, v in okapi.idf.items()})
print("Okapi 점수:", okapi_scores, "-> 0보다 큰 후보:", [s for s in okapi_scores if s > 0])
print("Index 점수:", mine.search(query))

**읽을 것.** Okapi는 점수가 음수여서 0보다 큰 것만 남기는 필터에서 후보가 **조용히 사라집니다.**

## 4. Dense와 코사인 유사도 (3분)

Dense 검색은 문장을 벡터로 바꿔 **뜻이 가까운지** 봅니다. 임베딩은 LLM과 **별개의 선택**이고 `intfloat/multilingual-e5-small`을 로컬에서 돌립니다.

In [ ]:
# [따라하기] 로컬 임베딩으로 코사인 유사도를 직접 계산한다 (무료, 첫 실행은 내려받기로 느림)
embeddings = make_embeddings()
vectors = np.array(embeddings.embed_documents(SENTENCES))
query_vector = np.array(embeddings.embed_query("대출 기간"))

print("벡터 모양:", vectors.shape, "| 길이:", np.linalg.norm(vectors, axis=1).round(3))
scores = vectors @ query_vector
for i in np.argsort(-scores):
    print(f"{scores[i]:.3f}", SENTENCES[i])

**관찰할 것.** 벡터 길이가 1이라 내적이 곧 코사인입니다. BM25가 빈손이던 질의에서 dense는 순위를 만들지만, 1위와 2위 차이가 작으면 불안정합니다.

**[바꿔보기]** 질의를 `점심 먹는 곳 여는 시간`으로 바꿉니다. **예측 먼저:** 영업 시간 문장이 1위일까요?

In [ ]:
# [바꿔보기] 바꿀 값: QUERY. 예측을 먼저 적고 실행하세요. (무료)
QUERY = "점심 먹는 곳 여는 시간"

scores = vectors @ np.array(embeddings.embed_query(QUERY))
order = np.argsort(-scores)
for i in order:
    print(f"{scores[i]:.3f}", SENTENCES[i])
print("1위와 2위의 점수 차이:", round(float(scores[order[0]] - scores[order[1]]), 3))
print("BM25 결과:", index.search(QUERY, limit=3))

**관찰할 것.** dense도 틀릴 수 있습니다. 비슷한 주제의 다른 문장이 위에 올 수 있습니다.

## 5. RRF: 점수가 아니라 순위 (3분)

BM25 점수와 코사인은 **단위가 다릅니다.** RRF(Reciprocal Rank Fusion)는 **순위**만 씁니다. 목록별로 `가중치 / (상수 + 순위)`를 더하며 상수는 60입니다.

In [ ]:
# [따라하기] RRF를 손으로 계산하고 rrf()와 맞춘다 (무료)
QUERY = "점심 먹는 곳 여는 시간"
bm25_rank = [cid for cid, _ in index.search(QUERY, limit=5)]
dense_scores = vectors @ np.array(embeddings.embed_query(QUERY))
dense_rank = [chunks[i].chunk_id for i in np.argsort(-dense_scores)]

table = {}
for ranking in (bm25_rank, dense_rank):
    for rank, chunk_id in enumerate(ranking, 1):
        table[chunk_id] = table.get(chunk_id, 0) + 1 / (60 + rank)
by_hand = sorted(table, key=lambda cid: (-table[cid], cid))

print("BM25 순위 :", bm25_rank)
print("Dense 순위:", dense_rank)
for cid in by_hand:
    print(f"{cid}  BM25 {bm25_rank.index(cid) + 1 if cid in bm25_rank else '-'}위, "
          f"Dense {dense_rank.index(cid) + 1}위 -> {table[cid]:.5f}")
print("손계산 순서 == rrf():", by_hand == rrf([bm25_rank, dense_rank]))

**관찰할 것.** 양쪽 목록 위쪽에 있는 문장이 합쳐진 순위에서 올라옵니다.

**[바꿔보기]** 상수와 가중치를 바꿉니다. `CONSTANT`를 1로, `WEIGHTS`를 `[0.0, 1.0]`로 두면 BM25의 몫이 사라집니다. **예측 먼저:** 순서가 바뀔까요?

In [ ]:
# [바꿔보기] 바꿀 값: CONSTANT와 WEIGHTS. 예측을 먼저 적고 실행하세요. (무료)
CONSTANT = 1
WEIGHTS = [0.0, 1.0]

print("기본값      :", rrf([bm25_rank, dense_rank]))
print("상수만 바꿈 :", rrf([bm25_rank, dense_rank], constant=CONSTANT))
print("가중치만 바꿈:", rrf([bm25_rank, dense_rank], weights=WEIGHTS))

In [ ]:
# [깨뜨리기] 잘못된 가중치와, 점수를 그냥 더하는 방식 (무료)
try:
    rrf([["a"]], weights=[1.0, 2.0])
except ValueError as error:
    print("ValueError:", error)

SPREAD_QUERY = "점심 먹는 곳 여는 시간"
bm25_all = dict(index.search(SPREAD_QUERY, limit=5))
bm25_values = np.array([bm25_all.get(c.chunk_id, 0.0) for c in chunks])
cosine_values = vectors @ np.array(embeddings.embed_query(SPREAD_QUERY))
print("BM25 점수 최대-최소:", round(float(bm25_values.max() - bm25_values.min()), 3))
print("코사인 최대-최소   :", round(float(cosine_values.max() - cosine_values.min()), 3))
summed = bm25_values + cosine_values
print("점수 합산 순서:", [chunks[i].chunk_id for i in np.argsort(-summed)])
print("BM25만의 순서 :", [chunks[i].chunk_id for i in np.argsort(-bm25_values, kind="stable")])
print("코사인만의 순서:", [chunks[i].chunk_id for i in np.argsort(-cosine_values)])

**읽을 것.** 합산 1위가 어느 쪽 1위와 같은지 보세요. 범위가 넓은 쪽이 정합니다.

## 6. 청킹 규칙, 메타데이터는 자격 (3분)

규칙은 `## ` 제목에서 나누고 **표는 나누지 않는 것**이며, 비교 대상은 글자 수로 자르는 `fixed_chunks`. 고객, 유효 기간, 승인 상태 같은 **메타데이터는 관련도가 아니라 자격**이라, 범위 필터는 점수를 매기기 **전에** 후보를 줄입니다.

In [ ]:
# [따라하기] 절 단위 조각과 고정 길이 조각의 시작과 끝을 본다 (무료)
documents = load_corpus()
v2 = documents["sla-a-v2"]

by_section = section_chunks(v2)
by_size = fixed_chunks(v2, 400, 80)
for name, pieces in (("절 단위", by_section), ("고정 400/80", by_size)):
    print(f"{name}: 조각 {len(pieces)}개, 잘린 표 {count_cut_tables(v2, pieces)}개")
    for c in pieces:
        print("  ", c.chunk_id, "|", repr(c.text[:24]), "...", repr(c.text[-18:]))

**관찰할 것.** 절 단위 조각은 제목으로 시작해 문장 끝에서 끝나고, 고정 길이 조각은 아무 데나 잘립니다.

**[바꿔보기]** 크기 `SIZE`를 150으로 줄입니다(겹침 `OVERLAP`은 30). **예측 먼저:** 잘린 표가 몇 개일까요?

In [ ]:
# [바꿔보기] 바꿀 값: SIZE와 OVERLAP. 예측을 먼저 적고 실행하세요. (무료)
SIZE, OVERLAP = 150, 30

small = fixed_chunks(v2, SIZE, OVERLAP)
print(f"조각 {len(small)}개, 잘린 표 {count_cut_tables(v2, small)}개")

In [ ]:
# [깨뜨리기] 150/30에서 머리글 없는 표 행 조각을 찾는다 (무료)
header = "| 월간 가용률 | 서비스 크레딧 | 계산 기준 |"
for c in small:
    if c.start > 0 and "|" in c.text and header not in c.text:
        print(c.chunk_id, repr(c.text))

**읽을 것.** 머리글 없는 조각에는 `20%`만 남아 **무엇의 20%인지** 알 수 없습니다.

## 7. 범위 필터 (3분)

`rank(질문, scope)`는 `scope`에 맞지 않는 문서의 조각을 점수 계산에서 **제외**합니다(`Document.applies`: 승인 상태, 고객, 기준일).

In [ ]:
# [따라하기] 범위 없이 / 범위로 같은 질문을 검색해 문서로 올려 본다 (무료)
retriever_bm25 = Retriever(documents)
QUESTION = "알파 서비스 크레딧 신청 기한은 언제인가요?"

def to_docs(chunk_ids):
    return list(dict.fromkeys(retriever_bm25.chunks[c].doc_id for c in chunk_ids))

without = to_docs(retriever_bm25.rank(QUESTION, None, limit=6))
scope = Scope("알파", date(2026, 9, 8))
within = to_docs(retriever_bm25.rank(QUESTION, scope, limit=6))
print("범위 없음:", without)
print("범위 있음:", within)
print("사라진 문서:", [d for d in without if d not in within])

**관찰할 것.** 초안, 다른 고객, 이미 끝난 버전이 사라집니다. **자격이 없어서** 빠졌습니다.

**[바꿔보기]** 기준일 `AS_OF`를 2026년 6월 30일로 바꿉니다. **예측 먼저:** 어느 문서가 남을까요?

In [ ]:
# [바꿔보기] 바꿀 값: AS_OF. 예측을 먼저 적고 실행하세요. (무료)
AS_OF = date(2026, 6, 30)

old = Scope("알파", AS_OF)
print("남은 문서:", to_docs(retriever_bm25.rank(QUESTION, old, limit=6)))
for doc_id in ("sla-a-v1", "sla-a-v2", "sla-a-amendment", "sla-a-draft"):
    doc = documents[doc_id]
    print(f"{doc_id:16} 유효 {doc.valid_from} ~ {doc.valid_until or '현재'}, 상태 {doc.status}, applies={doc.applies(old)}")

In [ ]:
# [깨뜨리기] 사후 필터와 사전 필터를 비교하고, 임베딩 없는 dense를 만들어 본다 (무료)
top2 = to_docs(retriever_bm25.rank(QUESTION, None, limit=2))
after = [d for d in top2 if documents[d].applies(scope)]
before = to_docs(retriever_bm25.rank(QUESTION, scope, limit=2))
print("상위 2개를 뽑은 뒤 거름 :", after, f"-> 후보 {len(after)}개")
print("거른 뒤 상위 2개를 뽑음 :", before, f"-> 후보 {len(before)}개")

try:
    Retriever(documents, mode="dense")
except ValueError as error:
    print("ValueError:", error)

**읽을 것.** 사후 필터는 자격 없는 문서가 상위를 차지한 뒤 걸러서 후보가 **모자랍니다.** 그래서 범위는 검색 전에 겁니다.

## 8. 측정: recall, 누락과 오인 (2분)

**recall**은 정답 문서 중 상위 k개 안에서 찾은 비율, **누락**은 못 찾은 것, **오인**은 범위 밖 문서를 가져온 것입니다.

**두 정답표는 검토 전 초안입니다**(`review_status: draft`). 틀린 행이 있어도 측정 뒤에 고쳐 점수를 맞추지 않습니다.

## 9. 어려운 질문으로 측정 (5분)

방식 세 가지(`bm25`, `dense`, `hybrid`)를 두 질문 모음에서 잽니다. 정답표는 **평가 셀에서만** 읽습니다.

In [ ]:
# [따라하기] 어려운 질문으로 두 질문 모음의 유형별 recall을 잰다 (무료, 모델 호출 0회)
cases_1, cases_2 = load_cases(ROOT / "data/cases_hard.jsonl"), load_cases(ROOT / "data/cases_hard2.jsonl")
cases = cases_1 + cases_2  # 두 모음의 id는 겹치지 않는다
gold = {**load_gold(ROOT / "data/gold_hard.jsonl"), **load_gold(ROOT / "data/gold_hard2.jsonl")}
SET_OF = {c["id"]: "hard" for c in cases_1} | {c["id"]: "hard2" for c in cases_2}
retrievers = {mode: Retriever(documents, embeddings=embeddings, mode=mode) for mode in ("bm25", "dense", "hybrid")}

results = {}  # (방식, k) -> 행 목록. 아래 셀들이 이 값을 읽는다.

def measure(k):
    for mode, retriever in retrievers.items():
        results[(mode, k)] = evaluate_retrieval(retriever, cases, gold, k=k)

def show(k):
    summaries = {mode: {s["type"]: s for s in recall_by_type(results[(mode, k)])} for mode in retrievers}
    print(f"k={k}   " + "".join(f"{mode:>9}" for mode in retrievers))
    for case_type in summaries["bm25"]:
        cells = [summaries[mode][case_type]["recall"] for mode in retrievers]
        n = summaries["bm25"][case_type]["cases"]
        print(f"{case_type:14}" + "".join(f"{'-' if v is None else format(v, '.2f'):>9}" for v in cells) + f"   (질문 {n}개)")
    for mode in retrievers:
        for row in results[(mode, k)]:
            if row["missing_docs"] or row["out_of_scope_docs"]:
                print(f"  실패 [{mode}] {row['case_id']}: 찾음={row['retrieved_docs']} 누락={row['missing_docs']} 오인={row['out_of_scope_docs']}")

measure(2)
show(2)

**전체 표.** `k`를 1, 2, 3으로 재고 **MRR**을 더합니다. MRR은 기대 문서가 처음 나온 순위의 역수(1, 0.5, ... 상위 k개 밖이면 0)의 평균이고 k=3 결과로 셉니다.

In [ ]:
# [따라하기] 두 모음에서 recall@1,2,3과 MRR을 방식별로 잰다 (무료, 모델 호출 0회)
from rag_lab.evaluation import mrr

for k in (1, 2, 3):
    measure(k)

def subset(rows, name):
    return rows if name == "전체" else [r for r in rows if SET_OF[r["case_id"]] == name]

def mean_recall(rows):
    values = [r["recall"] for r in rows if r["recall"] is not None]
    return (sum(values) / len(values) if values else None), len(values)

def fmt(value):
    return "-" if value is None else f"{value:.3f}"

for name in ("hard", "hard2", "전체"):
    n_all = len(subset(results[("bm25", 1)], name))
    n_scored = mean_recall(subset(results[("bm25", 1)], name))[1]
    print(f"[{name}] 질문 {n_all}개, 정답 문서가 있어 채점한 질문 {n_scored}개")
    print(f"{'방식':8}{'recall@1':>10}{'recall@2':>10}{'recall@3':>10}{'MRR@3':>9}{'누락 행(k=2)':>14}")
    for mode in retrievers:
        recalls = [mean_recall(subset(results[(mode, k)], name))[0] for k in (1, 2, 3)]
        m = mrr(subset(results[(mode, 3)], name))[0]
        failed = sum(1 for r in subset(results[(mode, 2)], name) if r["missing_docs"])
        print(f"{mode:8}" + "".join(f"{fmt(v):>10}" for v in recalls) + f"{fmt(m):>9}{failed:>14}")

print("\n누락이 난 행 (k=2, 지운 행 없음)")
for mode in retrievers:
    for r in results[(mode, 2)]:
        if r["missing_docs"]:
            print(f"  [{mode}] {r['case_id']} ({SET_OF[r['case_id']]}): 찾음={r['retrieved_docs']} 누락={r['missing_docs']}")

In [ ]:
# 결론 (표지 없음): 위에서 잰 값만 읽어 쓴다. 정답표를 다시 읽지 않는다.
KEYS = [("recall@1", lambda m, name: mean_recall(subset(results[(m, 1)], name))[0]),
        ("recall@2", lambda m, name: mean_recall(subset(results[(m, 2)], name))[0]),
        ("recall@3", lambda m, name: mean_recall(subset(results[(m, 3)], name))[0]),
        ("MRR@3", lambda m, name: mrr(subset(results[(m, 3)], name))[0])]
winners_by_set = {}
for name in ("hard", "hard2", "전체"):
    scored = mean_recall(subset(results[("bm25", 1)], name))[1]
    print(f"[{name}] 질문 {len(subset(results[('bm25', 1)], name))}개 중 채점 {scored}개. 정답표는 검토 전 초안입니다.")
    for label, fn in KEYS:
        values = {m: fn(m, name) for m in retrievers}
        best = max(round(v, 9) for v in values.values())
        top = [m for m, v in values.items() if round(v, 9) == best]
        winners_by_set[(name, label)] = tuple(top)
        verdict = "세 방식 동점" if len(top) == len(values) else (f"공동 1위(동점) {top}" if len(top) > 1 else f"1위 {top[0]}")
        print(f"  {label:9} {verdict}  " + str({m: round(v, 3) for m, v in values.items()}))

disagree = [label for label, _ in KEYS if winners_by_set[("hard", label)] != winners_by_set[("hard2", label)]]
print("두 모음에서 1위 집합이 다른 지표:", disagree or "없음")
print("누락 행 수 (k=2):", {m: sum(1 for r in results[(m, 2)] if r["missing_docs"]) for m in retrievers})
print("범위 밖 문서(오인) 총합:", sum(len(r["out_of_scope_docs"]) for rows in results.values() for r in rows))
if all(len(winners_by_set[("전체", label)]) == len(retrievers) for label, _ in KEYS):
    print("전체 표본은 방식을 가려내지 못합니다. 모든 지표에서 세 방식이 같습니다.")
elif len({winners_by_set[("전체", label)] for label, _ in KEYS}) == 1:
    print("전체 표본에서는 모든 지표의 1위가 같습니다:", list(winners_by_set[("전체", "recall@1")]), "다만 질문 수가 적고 정답표가 초안이라 한 행이 순위를 바꿀 수 있습니다.")
else:
    print("전체 표본에서 지표별 승자가 갈립니다. 다만 질문 수가 적고 정답표가 초안이라 한 행이 순위를 바꿀 수 있습니다.")

In [ ]:
# [따라하기] 상위 1개만 찾고 related_docs와 read_more를 본다 (무료)
case = next(c for c in cases if c["id"] == "para-claim-window")
context = RetrievalContext(retrievers["hybrid"], Scope(case["entity"], date.fromisoformat(case["as_of"])))

found = context.search(case["question"], limit=1)
print("찾은 문서:", [e["doc_id"] for e in found["evidence"]], "| related_docs:", found["related_docs"])

for doc_id in found["related_docs"][:1]:
    more = context.read_more(doc_id)
    print("읽은 문서:", doc_id, "|", more["evidence"]["quote"][:80].replace("\n", " / "))

**관찰할 것.** 상위 1개에 없던 문서가 `related_docs`에 있고, 읽으면 사실이 나옵니다. 이 읽기는 문서 recall에 **보이지 않습니다.**

## 10. 넓은 후보와 원 질문 재순위 (7분)

**후보 깊이가 recall의 천장입니다.** 재순위는 후보 안만 줄 세우므로 컷 밖의 정답은 되살리지 못합니다. `hybrid`는 목록마다 18개에서 자르고(`rag_lab/retrieval.py:16`), `fusion`은 범위 안 조각 전부의 점수를 dense .7로 합칩니다(`rag_lab/retrieval.py:88`). 올거나이즈 `hold`(1회)에서 RRF 후보 18개 → 융합 + 재순위 후보 50개로 MRR@10은 .692 → .766입니다.

AutoRAG 인용은 https://github.com/Marker-Inc-Korea/AutoRAG 의 커밋 `0c39052`(v2.13.4)입니다. 흐름: 분해 → 질의별 검색 → 번갈아 합침(`src/agent/agent.ts:3120`) → **원 질문** 기준 재순위(`:3145`) → 상위 25개(`src/retrieval/rerank.ts:42`). 이 저장소에서는 `retrieve`(`rag_lab/pipeline.py:63`) 한 함수입니다. 아래는 수업 문서 48문항으로 다시 잽니다. 정답 문서가 범위 밖인 행은 `범위 안`에서 뺍니다.

In [ ]:
# [따라하기] 수업 문서 48문항에서 hybrid와 fusion을 같은 표로 잰다 (무료, 모델 호출 0회)
all_cases = load_cases() + cases  # 기본 12문항 + 어려운 두 모음
all_gold = {**load_gold(), **gold}
fusion = Retriever(documents, embeddings=embeddings, mode="fusion")

def top_docs(run, k=3):
    return list(dict.fromkeys(fusion.chunks[c].doc_id for c in run.chunk_ids))[:k]

def case_scope(case):
    return Scope(case["entity"], date.fromisoformat(case["as_of"]))

NO_EXPECTED = [c["id"] for c in all_cases if not all_gold[c["id"]]["expected_docs"]]
# 정답 문서가 범위 밖이면 어떤 구성도 찾을 수 없다. 지우지 않고 따로 센다.
OUT_OF_SCOPE = [c["id"] for c in all_cases if c["id"] not in NO_EXPECTED
                and not all(documents[d].applies(case_scope(c)) for d in all_gold[c["id"]]["expected_docs"])]

def measure_all(name, retriever, rank=None):
    rows = {k: evaluate_retrieval(retriever, all_cases, all_gold, k=k, rank=rank) for k in (1, 2, 3)}
    for label, excluded in (("전체", []), ("범위 안", OUT_OF_SCOPE)):
        kept = {k: [r for r in rows[k] if r["case_id"] not in excluded] for k in rows}
        recalls = [fmt(mean_recall(kept[k])[0]) for k in (1, 2, 3)]
        mean_rr, scored = mrr(kept[3])
        print(f"{name:12} {label:5} r@1 {recalls[0]}  r@2 {recalls[1]}  r@3 {recalls[2]}  MRR@3 {fmt(mean_rr)}  (채점 {scored}문항)")
    return rows

def compare(before, after, k=3):
    pairs = [(b, a) for b, a in zip(before[k], after[k]) if b["expected_docs"]]
    gains = [(a["case_id"], mrr([a])[0] - mrr([b])[0], b["retrieved_docs"], a["retrieved_docs"]) for b, a in pairs]
    for title, picked in (("좋아진 행", [g for g in gains if g[1] > 0]), ("나빠진 행", [g for g in gains if g[1] < 0])):
        print(f"{title} {len(picked)}개")
        for case_id, change, was, now in picked:
            print(f"  {case_id}: 역순위 {change:+.2f}  {was} -> {now}")
    print("순위가 같은 행", sum(g[1] == 0 for g in gains), "개")

sizes = [len(fusion.rank(c["question"], case_scope(c), limit=len(fusion.chunks))) for c in all_cases]
print(f"전체 조각 {len(fusion.chunks)}개, 범위 안 조각 {min(sizes)}~{max(sizes)}개, 평균 {sum(sizes) / len(sizes):.1f}개")
print(f"문항 {len(all_cases)}개 = 채점 {len(all_cases) - len(NO_EXPECTED)}개 + 정답 문서 없음 {len(NO_EXPECTED)}개 {NO_EXPECTED}")
print(f"정답 문서가 범위 밖인 행 {len(OUT_OF_SCOPE)}개: {OUT_OF_SCOPE}")
hybrid_rows = measure_all("hybrid", retrievers["hybrid"])
fusion_rows = measure_all("fusion", fusion)
compare(hybrid_rows, fusion_rows)

**관찰할 것.** 수업 문서는 범위를 걸면 출력된 수만큼만 후보가 남아 18과 50의 차이가 나타나지 않습니다. 그 차이는 조각이 많은 말뭉치의 이야기입니다.

교차 인코더 `make_reranker`는 질문과 조각을 **한 쌍으로** 넣어 점수를 냅니다(`rag_lab/rerank.py:29`). **예측 먼저:** 1위 적중과 r@3 중 어느 쪽이 더 많이 오를까요?

In [ ]:
# [따라하기] 교차 인코더 재순위 전후를 같은 문항에서 잰다 (무료, 로컬 모델, 첫 실행은 내려받기로 느림)
reranker = make_reranker()
warnings_seen = []

def make_rank(pool):
    memo = {}  # evaluate_retrieval이 k마다 다시 부르므로 질문당 한 번만 재순위한다.
    def rank(question, scope):
        if (question, scope) not in memo:
            run = retrieve(question, retriever=fusion, scope=scope, reranker=reranker, pool=pool)
            warnings_seen.extend(run.warnings)
            memo[question, scope] = run.chunk_ids
        return memo[question, scope]
    return rank

started = perf_counter()
rerank_rows = measure_all("fusion+재순위", fusion, make_rank(50))
print(f"문항당 {(perf_counter() - started) / len(all_cases):.2f}초, 재순위 경고 {len(warnings_seen)}건")
print("\n재순위 전과 비교 (fusion -> fusion+재순위)")
compare(fusion_rows, rerank_rows)

print("\n놓친 행 (k=3, 지운 행 없음)")
for row in rerank_rows[3]:
    if row["missing_docs"]:
        reason = "범위 밖: 어느 구성으로도 못 찾음" if row["case_id"] in OUT_OF_SCOPE else "범위 안 실패"
        print(f"  [{reason}] {row['case_id']}: 찾음={row['retrieved_docs']} 누락={row['missing_docs']}")
print("1위에서 놓친 행 (k=1)")
for row in rerank_rows[1]:
    if row["missing_docs"]:
        print(f"  {row['case_id']}: 찾음={row['retrieved_docs']} 누락={row['missing_docs']}")

**관찰할 것.** 재순위는 찾은 것을 위로 올립니다. r@1과 MRR이 오르고 r@3은 같습니다. 범위 밖 행은 `전체`에 0점으로 들어 있습니다.

**[바꿔보기]** 후보 수 `POOLS`를 바꿔 천장을 직접 봅니다. **예측 먼저:** 후보가 1개면 r@3이 얼마일까요? 몇 개부터 값이 더 안 오를까요?

In [ ]:
# [바꿔보기] 바꿀 값: POOLS(재순위에 넘길 후보 수). 예측을 먼저 적고 실행하세요. (무료, 약 1분)
POOLS = [1, 2, 5, 18]

for pool in POOLS:
    measure_all(f"후보 {pool}개", fusion, make_rank(pool))
print("비교: 후보 50개는 바로 위 셀의 fusion+재순위 표입니다.")

In [ ]:
# [깨뜨리기] 재순위 모델이 실패하면 검색이 멈출까? (무료)
class BrokenReranker:
    def score(self, query, texts):
        raise RuntimeError("재순위 모델 응답 없음")

probe = next(c for c in all_cases if c["id"] == "table-ratio-max")
broken = retrieve(probe["question"], retriever=fusion, scope=case_scope(probe), reranker=BrokenReranker())
print("경고:", broken.warnings)
print("검색 순서 그대로:", broken.chunk_ids == fusion.rank(probe["question"], case_scope(probe), limit=25))

**읽을 것.** 실패해도 검색 순서를 유지하고 경고를 남깁니다. AutoRAG도 같습니다(`src/agent/agent.ts:3182`).

## 11. Jev 관문: 분해와 출처 보강은 기본으로 켠다 (6분)

**분해가 핵심 기능인 이유.** "장애 대응 절차와 로그 보관 기간"처럼 **사실 여러 개를 묶은 질문**은 한 번 검색하면 한쪽 사실의 조각이 밀릴 수 있어 AutoRAG는 질의 여러 개로 나눠 합칩니다. 이 수업도 분해를 **기본으로 켜되**, Decision API가 나눌지 판단합니다. `needs_decomposition`(`rag_lab/gates.py:93`)의 확률 P가 문턱값 0.5(`rag_lab/gates.py:18`) 이상이면 `decompose_question`을 부릅니다(`rag_lab/phases.py:54`).

**Jev를 읽는 법.** P는 보정된 확률이 아닙니다(`rag_lab/decisions.py:3`). 문턱값 0.5는 라벨로 정한 값이 아닌 코드 규칙이고, 공급자마다 보정에 대한 설명이 다릅니다. **질문 문장은 공급자에게 그대로 전송됩니다.** 판단이 실패하면 나누지 않습니다.

In [ ]:
# [따라하기] 수업 문서 질문 세 개를 분해 관문에 넣는다 (Jev 3회, Claude 0회)
decider = make_decider()
GATE_IDS = ["cross-ops-and-log-retention", "simple-deadline-current", "table-ratio-max"]

print(f"[분해 관문] Decision API(Jev) 호출 {len(GATE_IDS)}회, Claude 호출 0회입니다. 질문 문장이 공급자로 전송됩니다. 진행합니다.")
for case_id in GATE_IDS:
    question = next(c for c in all_cases if c["id"] == case_id)["question"]
    gate = needs_decomposition(decider, question)
    print(f"{case_id:28} backend={gate.backend} model={gate.model} P={gate.probability} "
          f"문턱값={gate.threshold} -> 분해 {'예' if gate.decision else '아니요'}" + (f" (대체 {gate.fallback})" if gate.fallback else ""))

**관찰할 것.** 저자의 48문항 훑기(1회)에서는 5문항(.104)만 열렸고 `cross-ops-and-log-retention`(.72)이 그중 하나입니다(`docs/retrieval_benchmarks.md`). 저자의 이번 실행(1회)은 `cross-ops-and-log-retention` .72로 열렸고 `simple-deadline-current` .14, `table-ratio-max` .22는 닫혔습니다. P는 호출마다 달라집니다.

공급자 문서는 docs.typesafe.ai, developers.openai.com/api/docs/guides/decisions입니다. 아래는 이 저장소가 코드로 맞춘 것만 적습니다.

| 공급자(`rag_lab/decisions.py`) | 참/거짓 질문 | 기준 문장 | 응답 |
|---|---|---|---|
| `TypeSafeBackend`, `OpenRouterBackend` | `noul` | `criteria` | 이름을 키로 한 map |
| `OpenAIDecisionsBackend` | `predicate` | `instructions`에 합침 | 목록 |

새 공급자는 `_HttpBackend`를 상속한 클래스 하나(`_body`, `_verdicts`)를 쓰고 `BACKENDS`(`rag_lab/decisions.py:177`)에 등록하면 됩니다. `make_decider`(`rag_lab/decisions.py:180`)는 키가 있는 첫 공급자를 쓰고 없으면 멈춥니다.

**[바꿔보기]** 열린 질문을 분해해 합칩니다. **예측 먼저:** 최종 상위 문서가 달라질까요? `QUESTION_ID`를 `simple-deadline-current`로 바꾸면?

In [ ]:
# [바꿔보기] 바꿀 값: QUESTION_ID. 관문이 열리면 분해해 후보를 합치고 원 질문으로 재순위한다 (Jev 1회, 열리면 Claude 1회)
QUESTION_ID = "cross-ops-and-log-retention"

class RecordingReranker:
    """재순위에 들어간 질문을 기록만 하고 점수는 reranker에 맡긴다."""
    def __init__(self, inner):
        self.inner, self.queries = inner, set()

    def score(self, query, texts):
        self.queries.add(query)
        return self.inner.score(query, texts)

probe = next(c for c in all_cases if c["id"] == QUESTION_ID)
print("[분해 관문] Decision API(Jev) 호출 1회, Claude 호출은 관문이 열릴 때만 1회입니다. 진행합니다.")
gate = needs_decomposition(decider, probe["question"])
print(f"관문: P={gate.probability} 문턱값={gate.threshold} -> {'열림' if gate.decision else '닫힘'}")

split_queries = []
if gate.decision:
    announce("질문 분해", model_calls=1)
    split_queries = decompose_question(make_model(), probe["question"])

recorder = RecordingReranker(reranker)
plain = retrieve(probe["question"], retriever=fusion, scope=case_scope(probe), reranker=recorder)
split = retrieve(probe["question"], retriever=fusion, scope=case_scope(probe), reranker=recorder, queries=split_queries or None)
print("질문:", probe["question"], "| 기대 문서:", all_gold[QUESTION_ID]["expected_docs"])
print("분해 질의:", split_queries or "(나누지 않음)")
allowed = len(fusion.rank(probe["question"], case_scope(probe), limit=len(fusion.chunks)))
print(f"검색한 질의 {len(split.queries)}개 -> 합친 후보 {split.candidates}개 (전체 조각 {len(fusion.chunks)}개 중 범위가 허용한 조각 {allowed}개)")
print(f"원 질문만 : 후보 {plain.candidates}개 -> {top_docs(plain)}")
print(f"분해 합침 : 후보 {split.candidates}개 -> {top_docs(split)}")
print("재순위에 들어간 질문이 원 질문 하나뿐인가:", recorder.queries == {probe["question"]})

**관찰할 것.** 관문의 P, 합친 후보 수, 재순위에 들어간 질문이 원 질문뿐인지를 보세요. 분해 질의는 후보를 넓힐 뿐 점수는 원 질문으로만 매깁니다. 저자의 이번 실행은 질의 6개를 검색해도 합친 후보가 원 질문만과 같은 15개였고 상위 문서도 같았습니다.

**수업 문서에서는 분해와 출처 보강이 재순위 결과를 구조적으로 바꿀 수 없습니다.** 후보 50개가 범위 안 조각(최대 32개)보다 많아 전부 후보이고 재순위가 그 전체를 다시 줄 세우기 때문입니다.

| 말뭉치, 정답 모양 | 비교 | 결과 |
|---|---|---|
| 올거나이즈 `tune`, 한 페이지 | 게이트 분해 | MRR@10 −.010 (−.029~.000) |
| 올거나이즈 `hold`, 한 페이지 | 항상 분해 | 5위 적중 .960 → .980, MRR +.004 (.000~.010) |
| Ko-StrategyQA `dev`, 여러 문단 | 분해 후보, nDCG@10 | +.011 (.004~.018) |
| 같은 말뭉치 | 최고 점수 재순위 병행 | +.017 (.006~.028) |

이득은 질문과 정답의 모양에 달렸습니다. 정답이 한 페이지인 시험은 여러 사실을 묶은 질문의 이득을 보일 수 없고, 여러 문단인 시험에서는 구간이 0 위였습니다. 다만 그 질문은 평균 22.6자로 짧고 마지막 행의 재순위는 채택하지 않았습니다. 켜 두는 것은 AutoRAG를 따르는 설계입니다.

**출처 보강**(`corroborate`, `rag_lab/retrieval.py:111`)은 같은 출처의 조각이 많거나 dense와 BM25가 둘 다 찾은 조각에 가산점(상한 .3)을 주며 기본값입니다. 재순위 없이 융합 순서만 켬과 끔으로 비교합니다. **예측 먼저:** 순서가 바뀌는 조각이 있을까요?

In [ ]:
# [바꿔보기] 바꿀 값: CORRO_ID. 예측을 먼저 적고 실행하세요. 출처 보강 켬/끔의 융합 순서를 재순위 없이 비교한다 (무료)
CORRO_ID = "simple-amendment-scope"
fusion_off = Retriever(documents, embeddings=embeddings, mode="fusion", corroboration=False)

probe = next(c for c in all_cases if c["id"] == CORRO_ID)
on = fusion.rank(probe["question"], case_scope(probe), limit=8)
off = fusion_off.rank(probe["question"], case_scope(probe), limit=8)
print("질문:", probe["question"])
for rank, chunk_id in enumerate(on, 1):
    off_rank = off.index(chunk_id) + 1 if chunk_id in off else "-"
    print(f"켬 {rank}위 {chunk_id:30} 끔 {off_rank}위" + ("   <- 움직임" if off_rank != rank else ""))
print("문서 순서 켬:", list(dict.fromkeys(fusion.chunks[c].doc_id for c in on))[:3], "| 끔:", list(dict.fromkeys(fusion.chunks[c].doc_id for c in off))[:3])

differ = [c["id"] for c in all_cases
          if fusion.rank(c["question"], case_scope(c), limit=len(fusion.chunks)) != fusion_off.rank(c["question"], case_scope(c), limit=len(fusion.chunks))]
print(f"재순위 없는 융합 순서가 켬과 끔에서 다른 질문: {len(differ)}개 / {len(all_cases)}개")

**관찰할 것.** 저자의 이번 실행은 조각 8개 중 7개가 움직였고 재순위 없는 융합 순서가 48문항 중 46개에서 달랐습니다. 단, 재순위를 붙이면 수업 문서 48문항의 문서 순위는 모두 같았습니다(위 구조적 이유). 효과는 올거나이즈 `tune`에서 쟀습니다.

| MRR@10, 끔 → 켬 | 재순위 정상 | 재순위 실패 |
|---|---|---|
| 출처=쪽 | +.008 (−.005~.029) | −.057 (−.090~−.029) |
| 출처=파일 | +.009 (−.001~.029) | −.015 (−.039~−.001) |

재순위가 정상이면 측정 가능한 변화가 없고, 실패하면 켠 쪽이 낮았습니다(파일 출처는 폭이 4분의 1쯤이지만 여전히 낮음). **"도움이 된다"는 숫자는 없습니다.** 켜 두는 것은 AutoRAG를 따르는 저자의 설계 결정입니다. 원인은 분석하지 않았습니다.

## 12. AutoRAG와 다른 점 (3분)

| 항목 | AutoRAG | 이 수업 | 수치(저자 1회) |
|---|---|---|---|
| 토크나이저 | 낱말 + 형태소(`docs/minsync-setup.md:287`) | 형태소만(`rag_lab/retrieval.py:22`) | BM25 단독 nDCG .581 대 .558(낱말+형태소), Ko-StrategyQA |
| 출처 보강 | 같은 출처에 가산(`src/retrieval/merger.ts:22`) | 켬(`Retriever`의 기본값) | 정상이면 변화 없고 실패하면 낮아짐(위 표) |
| 범위 | 경로 범위뿐(`src/retrieval/types.ts:33`) | 고객·기준일 사전 필터 | 아래 셀 |

**벤치마크 승자가 실제 PDF로 넘어가지 않았습니다.** 질문별 최고 점수 재순위는 Ko-StrategyQA에서 1위(nDCG .835)였지만 올거나이즈 `tune`에서는 1위 적중이 낮아(.714 대 .762) 원 질문 기준을 남겼습니다. 토크나이저도 근거가 Ko-StrategyQA 하나라 같은 위험이 있습니다(AutoRAG 문서는 반대 값, `docs/minsync-setup.md:300`).

In [ ]:
# [깨뜨리기] 범위 필터를 끄고 범위 밖 문서가 얼마나 올라오는지 센다 (무료)
for label, use_scope in (("범위 켬", True), ("범위 끔", False)):
    scope_rows = evaluate_retrieval(fusion, all_cases, all_gold, k=3, use_scope=use_scope)
    wrong = [r for r in scope_rows if r["out_of_scope_docs"]]
    print(f"{label}: 상위 3개에 범위 밖 문서가 든 행 {len(wrong)}개 / {len(scope_rows)}개")
    print("  이 행들:", [r["case_id"] for r in wrong])

**읽을 것.** 범위를 끄면 오인이 생깁니다(저자 0행 → 45행). 재순위는 자격을 보지 않아 막지 못합니다.

## 13. PDF 쪽 판독 (5분)

실제 PDF에서 가장 큰 개선은 이것이었습니다. 차트와 스캔 쪽에는 글자층이 없어 **색인할 글자가 없습니다.** 표본 `data/pdf/credit-trend-chart.pdf`는 차트 이미지 한 장이고 **수치는 지어낸 예시**입니다.

In [ ]:
# [따라하기] 글자층이 없는 쪽은 검색에 잡히지 않는다 (무료)
PDF_PATH = ROOT / "data/pdf/credit-trend-chart.pdf"
PAGE_QUERY = "서비스 크레딧 신청 건수가 가장 많은 분기"

def page_search(text):
    return BM25Index([Chunk("p1", "credit-trend-chart", "", text, 0, len(text), 1, 1)]).search(PAGE_QUERY)

with pymupdf.open(PDF_PATH) as pdf:
    layer = pdf[0].get_text()
    print("쪽 수:", pdf.page_count, "| 글자층:", repr(layer), "| needs_reading:", needs_reading(pdf[0]))
print("글자층 검색 결과:", page_search(layer))

In [ ]:
# [따라하기] Claude가 쪽을 읽어 검색할 수 있는 글자로 옮긴다. 유료 1회(이미지 한 장).
announce("PDF 쪽 판독", model_calls=1)
transcript = read_page(make_model(), PDF_PATH, 1)
print(transcript)
print("\n판독 글 검색 결과:", page_search(transcript))

In [ ]:
# [깨뜨리기] 없는 쪽을 달라고 한다. 모델을 부르기 전에 멈춘다 (무료)
try:
    read_page(make_model(), PDF_PATH, 2)
except ValueError as error:
    print("ValueError:", error)

**읽을 것.** 판독 글로 후보를 찾습니다. **판독 글의 정확성은 검증하지 않았습니다.** 인용하려면 원본 쪽 대조가 따로 필요합니다.

저자가 올거나이즈 `hold`에서 잰 값(309쪽 선택, $3.89, 5분 26초)은 MRR@10 .766 → .873, 이미지 유형 5위 적중 14/24 → 21/24입니다.

## 직접 해보기 (5분)

아래 세 셀은 **예시가 채워져** 있어서 그대로 실행됩니다. `MY_`로 시작하는 값을 내 것으로 바꿔 다시 실행하세요.

In [ ]:
# [내 것 만들기] 내 문장 다섯 개와 내 질의로 토큰, BM25, 코사인, RRF를 한 번에 본다 (무료)
MY_SENTENCES = ["팀 회의는 매주 화요일 오전 10시에 합니다.", "회의록은 회의 당일 저녁까지 공유 폴더에 올립니다.",
                "점심 회식은 팀장 승인을 받은 뒤에 예약합니다.", "휴가는 사흘 전까지 캘린더에 먼저 등록합니다.",
                "긴급 장애는 당직 담당자에게 전화로 먼저 알립니다."]
MY_QUERY = "회식 하려면 누구한테 허락받아요?"

if len(MY_SENTENCES) < 2 or not MY_QUERY.strip():
    print("MY_SENTENCES에 문장을 둘 이상, MY_QUERY에 질의를 채우세요.")
else:
    mine = [Chunk(f"m{i}", "mine", "", s, 0, len(s), 1, 1) for i, s in enumerate(MY_SENTENCES)]
    my_index = BM25Index(mine)
    for s in MY_SENTENCES:
        print("토큰:", tokenize(s))
    bm25_hits = my_index.search(MY_QUERY, limit=len(mine))
    cos = np.array(embeddings.embed_documents(MY_SENTENCES)) @ np.array(embeddings.embed_query(MY_QUERY))
    dense_hits = [mine[i].chunk_id for i in np.argsort(-cos)]
    print("BM25  :", bm25_hits)
    print("코사인:", [(mine[i].chunk_id, round(float(cos[i]), 3)) for i in np.argsort(-cos)])
    fused = rrf([[c for c, _ in bm25_hits], dense_hits])
    print("RRF   :", [(c, MY_SENTENCES[int(c[1:])]) for c in fused[:3]])

In [ ]:
# [내 것 만들기] 내 질문이 기대한 문서를 네 방식의 상위 2개에서 찾는지 본다 (무료)
MY_QUESTION = "출장 가서 하룻밤 자면 숙박비를 얼마까지 낼 수 있나요?"
MY_EXPECTED_DOC = "travel"
MY_ENTITY = "공통"
MY_AS_OF = "2026-09-08"

if not MY_QUESTION.strip() or MY_EXPECTED_DOC not in documents:
    print("MY_QUESTION을 채우고, MY_EXPECTED_DOC은 다음 중 하나로 채우세요:", list(documents))
else:
    my_scope = Scope(MY_ENTITY, date.fromisoformat(MY_AS_OF))
    for mode, retriever in retrievers.items():
        ids = retriever.rank(MY_QUESTION, my_scope, limit=len(retriever.chunks))
        top2 = list(dict.fromkeys(retriever.chunks[c].doc_id for c in ids))[:2]
        print(f"{mode:7} 상위 2개={top2}  기대 문서 {'찾음' if MY_EXPECTED_DOC in top2 else '못 찾음'}")
    top2 = top_docs(retrieve(MY_QUESTION, retriever=fusion, scope=my_scope, reranker=reranker), k=2)
    print(f"재순위   상위 2개={top2}  기대 문서 {'찾음' if MY_EXPECTED_DOC in top2 else '못 찾음'}")

In [ ]:
# [내 것 만들기] 내 질문이 관문을 넘으면 decompose_question으로 나누고 원 질문으로 재순위한다 (Jev 1회, 열리면 Claude 1회)
MY_QUESTION = "출장 가서 하룻밤 자면 숙박비를 얼마까지 낼 수 있나요?"

if not MY_QUESTION.strip():
    print("MY_QUESTION에 질문을 채우세요.")
else:
    my_scope = Scope(MY_ENTITY, date.fromisoformat(MY_AS_OF))
    print("[내 질문의 관문] Decision API(Jev) 호출 1회, Claude 호출은 관문이 열릴 때만 1회입니다. 진행합니다.")
    my_gate = needs_decomposition(decider, MY_QUESTION)
    print(f"관문: P={my_gate.probability} 문턱값={my_gate.threshold} -> {'열림' if my_gate.decision else '닫힘'}")
    my_queries = []
    if my_gate.decision:
        announce("내 질문의 분해", model_calls=1)
        my_queries = decompose_question(make_model(), MY_QUESTION)
    print("분해 질의:", my_queries or "(나누지 않음)")
    print("원 질문만:", top_docs(retrieve(MY_QUESTION, retriever=fusion, scope=my_scope, reranker=reranker), k=2))
    print("분해 합침:", top_docs(retrieve(MY_QUESTION, retriever=fusion, scope=my_scope, reranker=reranker,
                                        queries=my_queries or None), k=2))

## 확인 질문 (3분)

1. `식당`이 다섯 문장 중 두 문장에 있을 때 IDF를 말해 보세요.
2. 4절 [바꿔보기] 출력으로, 이 질의의 dense 1위를 인용 근거로 써도 되는지 판단하세요. 근거로 읽은 값은?
3. 7절 [깨뜨리기] 출력의 두 줄에서 후보를 잃은 문서와 자리를 잃은 단계는? 상위 몇 개를 뽑아도 사후 필터가 안전하지 않은 이유는?
4. 10절 `POOLS` 출력의 후보 1개, 2개일 때 범위 안 r@3을 읽고 재순위가 그 값을 못 넘는 이유를 쓰세요. 18개와 50개가 같아도 50개가 쓸모없다고 할 수 없는 말뭉치 조건은?
5. 12절의 .835와 (.714 대 .762)를 한 줄에 놓고 비교하면 안 되는 부분은? 순위가 뒤집힌 이유 후보 중 이 노트북이 측정한 것은?
6. 11절 관문 출력의 P(실행마다 다릅니다)를 '분해가 필요할 확률'로 읽으면 어디가 틀립니까? 회사 질문에 돌리기 전에 확인할 것은?
7. 11절 출력의 `합친 후보`와 `범위가 허용한 조각`으로, 후보 상한 50이 어떤 조건일 때 분해가 재순위 결과를 바꾸는지 부등식으로 쓰세요. 켜 둔 근거는 측정입니까, 설계 선택입니까?

## 정리 (2분)

- 근거 단위는 규칙으로 지키고, 메타데이터는 자격이므로 범위는 검색 전에 겁니다.
- 후보는 넓게, 재순위는 원 질문으로, 실패하면 검색 순서와 경고로. 후보 깊이가 천장입니다.
- 분해는 Decision API 관문 뒤에서, 출처 보강도 기본으로 켭니다. P는 보정되지 않은 값이고 문턱값은 코드 규칙입니다. 분해의 이득은 질문과 정답의 모양에 달렸고, 출처 보강은 재순위가 정상이면 변화가 없고 실패하면 낮았습니다. 켠 이유는 설계 선택입니다.
- 우열은 **측정**으로 정하고 실패한 행을 지우지 않습니다.
- 글자층 없는 쪽은 판독해야 찾지만 판독 글의 정확성은 따로 확인합니다. 다음 노트북은 이 검색을 운영형 Agent에 연결합니다.